In [9]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")

db_kopi = client["kopi_nusantara"]
koleksi_menu = db_kopi["menu"]


koleksi_menu.delete_many({})

data_menu = [
    # Kategori Kopi 
    {"nama_menu": "Espresso Single", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 50},
    {"nama_menu": "Americano Hot", "harga": 20000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 60},
    {"nama_menu": "Kopi Tubruk Gayo", "harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 40},
    {"nama_menu": "Caffè Latte", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 45},
    {"nama_menu": "Kopi Susu Aren", "harga": 23000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 100},
    {"nama_menu": "Cold Brew Concentrate", "harga": 28000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 30},
    
    # Kategori Makanan 
    {"nama_menu": "Nasi Goreng Rempah", "harga": 32000, "kategori": "Makanan", "pedas_level": 2},
    {"nama_menu": "Mie Goreng Jawa", "harga": 27000, "kategori": "Makanan", "pedas_level": 1},
    {"nama_menu": "Ayam Geprek Sambal Ijo", "harga": 25000, "kategori": "Makanan", "pedas_level": 3},
    {"nama_menu": "Roti Bakar Cokelat", "harga": 18000, "kategori": "Makanan", "pedas_level": 0},
    
    # Kategori Non-Kopi
    {"nama_menu": "Matcha Cream Latte", "harga": 26000, "kategori": "Non-Kopi", "rekomendasi": True},
    {"nama_menu": "Teh Tarik Nusantara", "harga": 15000, "kategori": "Non-Kopi", "rekomendasi": False}
]

koleksi_menu.insert_many(data_menu) Cell In[5], line 35
    ```[cite: 1]
    ^
SyntaxError: invalid syntax

InsertManyResult([ObjectId('6aeb6741dcb48cfea3a01a04'), ObjectId('6aeb6741dcb48cfea3a01a05'), ObjectId('6aeb6741dcb48cfea3a01a06'), ObjectId('6aeb6741dcb48cfea3a01a07'), ObjectId('6aeb6741dcb48cfea3a01a08'), ObjectId('6aeb6741dcb48cfea3a01a09'), ObjectId('6aeb6741dcb48cfea3a01a0a'), ObjectId('6aeb6741dcb48cfea3a01a0b'), ObjectId('6aeb6741dcb48cfea3a01a0c'), ObjectId('6aeb6741dcb48cfea3a01a0d'), ObjectId('6aeb6741dcb48cfea3a01a0e'), ObjectId('6aeb6741dcb48cfea3a01a0f')], acknowledged=True)

In [12]:
# 1. Kategori Kopi dengan tingkat_kepahitan > 3
print("1. Menu Kopi Kepahitan > 3 : ")
for m in koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f" - {m['nama_menu']} | Kepahitan: {m['tingkat_kepahitan']}")

# 2. Menu dengan harga antara Rp15.000 dan Rp30.000
print("\n2. Menu Harga Rp15.000 s/d Rp30.000 : ")
for m in koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
    print(f" - {m['nama_menu']}: Rp{m['harga']:,}")

# 3. Menu yang tidak memiliki field pedas_level
print("\n3. Menu Tanpa Field pedas_level : ")
for m in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f" - {m['nama_menu']} ({m['kategori']})")

1. Menu Kopi Kepahitan > 3 : 
 - Espresso Single | Kepahitan: 5
 - Americano Hot | Kepahitan: 4
 - Kopi Tubruk Gayo | Kepahitan: 5
 - Cold Brew Concentrate | Kepahitan: 4

2. Menu Harga Rp15.000 s/d Rp30.000 : 
 - Espresso Single: Rp18,000
 - Americano Hot: Rp20,000
 - Kopi Tubruk Gayo: Rp22,000
 - Caffè Latte: Rp25,000
 - Kopi Susu Aren: Rp23,000
 - Cold Brew Concentrate: Rp28,000
 - Mie Goreng Jawa: Rp27,000
 - Ayam Geprek Sambal Ijo: Rp25,000
 - Roti Bakar Cokelat: Rp18,000
 - Matcha Cream Latte: Rp26,000
 - Teh Tarik Nusantara: Rp15,000

3. Menu Tanpa Field pedas_level : 
 - Espresso Single (Kopi)
 - Americano Hot (Kopi)
 - Kopi Tubruk Gayo (Kopi)
 - Caffè Latte (Kopi)
 - Kopi Susu Aren (Kopi)
 - Cold Brew Concentrate (Kopi)
 - Matcha Cream Latte (Non-Kopi)
 - Teh Tarik Nusantara (Non-Kopi)


In [13]:
# 1. Menambahkan field status: "Tersedia" ke seluruh dokumen
koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})

# 2. Mengubah status 2 menu pilihan menjadi "Habis"
koleksi_menu.update_one({"nama_menu": "Cold Brew Concentrate"}, {"$set": {"status": "Habis"}})
koleksi_menu.update_one({"nama_menu": "Ayam Geprek Sambal Ijo"}, {"$set": {"status": "Habis"}})

UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

In [20]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, round

spark = SparkSession.builder \
    .appName("Tugas9-MongoDBSpark") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

dokumen_menu = list(koleksi_menu.find({}))

for d in dokumen_menu:
    d["_id"] = str(d["_id"])

df_spark_menu = spark.createDataFrame(dokumen_menu)

print("1. Schema Spark DataFrame")
df_spark_menu.printSchema()

print("\nData Spark DataFrame")
df_spark_menu.show(truncate=False)

print(" 2. Rata-Rata Harga per Kategori ")
df_spark_menu.groupBy("kategori").agg(
    round(avg("harga"), 2).alias("rata_rata_harga")
).show()

spark.stop()
print("SparkSession ditutup.")

1. Schema Spark DataFrame
root
 |-- _id: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- status: string (nullable = true)
 |-- stok: long (nullable = true)
 |-- tingkat_kepahitan: long (nullable = true)
 |-- pedas_level: long (nullable = true)
 |-- rekomendasi: boolean (nullable = true)


Data Spark DataFrame


+------------------------+-----+--------+----------------------+--------+----+-----------------+-----------+-----------+
|_id                     |harga|kategori|nama_menu             |status  |stok|tingkat_kepahitan|pedas_level|rekomendasi|
+------------------------+-----+--------+----------------------+--------+----+-----------------+-----------+-----------+
|6aeb6741dcb48cfea3a01a04|18000|Kopi    |Espresso Single       |Tersedia|50  |5                |NULL       |NULL       |
|6aeb6741dcb48cfea3a01a05|20000|Kopi    |Americano Hot         |Tersedia|60  |4                |NULL       |NULL       |
|6aeb6741dcb48cfea3a01a06|22000|Kopi    |Kopi Tubruk Gayo      |Tersedia|40  |5                |NULL       |NULL       |
|6aeb6741dcb48cfea3a01a07|25000|Kopi    |Caffè Latte           |Tersedia|45  |2                |NULL       |NULL       |
|6aeb6741dcb48cfea3a01a08|23000|Kopi    |Kopi Susu Aren        |Tersedia|100 |3                |NULL       |NULL       |
|6aeb6741dcb48cfea3a01a09|28000|

+--------+---------------+
|kategori|rata_rata_harga|
+--------+---------------+
|Non-Kopi|        20500.0|
| Makanan|        25500.0|
|    Kopi|       22666.67|
+--------+---------------+

SparkSession ditutup.


Tantangan utama saat mengonversi data dari MongoDB ke Spark DataFrame adalah perbedaan mendasar model data antara NoSQL (MongoDB) dan mesin pemrosesan terdistribusi terstruktur (Spark). MongoDB menggunakan skema fleksibel (*schema-less*), di mana setiap dokumen dalam satu *collection* dapat memiliki atribut atau field yang tidak seragam (misalnya `tingkat_kepahitan` yang hanya dimiliki oleh menu kategori Kopi, serta `pedas_level` yang hanya dimiliki oleh menu kategori Makanan). Sebaliknya, Spark DataFrame mewajibkan skema tabel yang kaku dan seragam untuk seluruh baris data.

Untuk mengatasi tantangan ini, kami memanfaatkan Pandas DataFrame sebagai *intermediate layer* saat konversi dari PyMongo ke Spark. Pandas secara otomatis menyatukan (*union*) seluruh kunci/field yang beredar di seluruh dokumen JSON. Atribut yang tidak dimiliki oleh dokumen tertentu akan otomatis diisi dengan nilai kosong (`NaN` atau `None`). Saat Pandas DataFrame diproses lebih lanjut oleh Spark melalui `createDataFrame()`, Spark secara cerdas memetakan nilai kosong tersebut menjadi nilai `null` sesuai tipe data kolom bersangkutan (seperti `double` atau `long`). Dengan mekanisme ini, struktur data terdistribusi tetap konsisten tanpa menghilangkan informasi spesifik antar-kategori menu.